# Notebook Objective

# 03 RUL Target and Preprocessing

This notebook transforms the exploratory findings from Notebook 02 into a reproducible preprocessing workflow for the NASA C-MAPSS dataset.

The main objectives are:

- load the selected C-MAPSS subset
- construct the raw Remaining Useful Life (RUL) target for training trajectories
- create a capped RUL target based on a piecewise linear degradation assumption
- prepare the true RUL information for the test set
- identify and remove constant or non-informative features
- define feature columns and target columns for later modeling
- create an engine-wise train-validation split to avoid data leakage
- prepare scaled feature versions for later machine learning and deep learning models
- validate that the processed datasets are structurally consistent

This notebook does not perform feature engineering with sliding windows and does not train models. These steps are handled in later notebooks.

# Imports and Project Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from src.data_loading import (
    load_cmapss_dataset,
    DATASET_IDS,
    SETTING_NAMES,
    SENSOR_NAMES,
)

from src.preprocessing import (
    add_rul_to_training_data,
    add_capped_rul,
    prepare_test_rul,
)

from src.config import load_config, flatten_config


# Load Dataset

The selected C-MAPSS subset is loaded from the raw data files. FD001 is used as the initial subset because it contains one operating condition and one fault mode.

In [3]:
config = load_config()
flat_config = flatten_config(config)

CURRENT_DATASET = flat_config["CURRENT_DATASET"]
RUL_CAP = flat_config["RUL_CAP"]
VALIDATION_SIZE = flat_config["VALIDATION_SIZE"]
RANDOM_STATE = flat_config["RANDOM_STATE"]
WINDOW_SIZE = flat_config["WINDOW_SIZE"]
WINDOW_STEP = flat_config["WINDOW_STEP"]

index_cols = flat_config["INDEX_COLS"]
target_cols = flat_config["TARGET_COLS"]
helper_cols = flat_config["HELPER_COLS"]
target_col = flat_config["TARGET_COL"]

model_config = config["models"]
conformal_config = config["conformal"]
simulation_config = config["simulation"]

if CURRENT_DATASET not in DATASET_IDS:
    raise ValueError(f"Unknown dataset: {CURRENT_DATASET}")

df_train_raw, df_test_raw, df_test_rul_raw = load_cmapss_dataset(CURRENT_DATASET)

print(f"Current dataset: {CURRENT_DATASET}")
print(f"Training data shape: {df_train_raw.shape}")
print(f"Test data shape:     {df_test_raw.shape}")
print(f"Test RUL shape:      {df_test_rul_raw.shape}")


Current dataset: FD001
Training data shape: (20631, 26)
Test data shape:     (13096, 26)
Test RUL shape:      (100, 1)


In [4]:
df_train_raw.head()

,engine,cycle,setting_1,setting_2,setting_3,T2 (Fan inlet temperature),T24 (LPC outlet temperature),T30 (HPC outlet temperature),T50 (LPT outlet temperature),P2 (Fan inlet pressure),P15 (bypass duct pressure),P30 (HPC outlet pressure),Nf (Physical fan speed),Nc (Physical core speed),epr (Engine pressure ratio (P50/P2)),Ps30 (Static pressure at HPC outlet),phi (Ratio of fuel flow to Ps30),NRf (Corrected fan speed),NRc (Corrected core speed),BPR (Bypass ratio),farB (Fuel-air ratio),htBleed (Bleed enthalpy),Nf_dmd (Demanded fan speed),PCNfr_dmd (Demanded corrected fan speed),W31 (HPT Cool air flow),W32 (LPT Cool air flow)
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.3,47.13,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.00,2388.06,9055.15,1.3,47.28,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [5]:
df_test_raw.head()

,engine,cycle,setting_1,setting_2,setting_3,T2 (Fan inlet temperature),T24 (LPC outlet temperature),T30 (HPC outlet temperature),T50 (LPT outlet temperature),P2 (Fan inlet pressure),P15 (bypass duct pressure),P30 (HPC outlet pressure),Nf (Physical fan speed),Nc (Physical core speed),epr (Engine pressure ratio (P50/P2)),Ps30 (Static pressure at HPC outlet),phi (Ratio of fuel flow to Ps30),NRf (Corrected fan speed),NRc (Corrected core speed),BPR (Bypass ratio),farB (Fuel-air ratio),htBleed (Bleed enthalpy),Nf_dmd (Demanded fan speed),PCNfr_dmd (Demanded corrected fan speed),W31 (HPT Cool air flow),W32 (LPT Cool air flow)
0,1,1,0.0023,0.0003,100.0,518.67,643.02,1585.29,1398.21,14.62,21.61,553.90,2388.04,9050.17,1.3,47.20,521.72,2388.03,8125.55,8.4052,0.03,392,2388,100.0,38.86,23.3735
1,1,2,-0.0027,-0.0003,100.0,518.67,641.71,1588.45,1395.42,14.62,21.61,554.85,2388.01,9054.42,1.3,47.50,522.16,2388.06,8139.62,8.3803,0.03,393,2388,100.0,39.02,23.3916
2,1,3,0.0003,0.0001,100.0,518.67,642.46,1586.94,1401.34,14.62,21.61,554.11,2388.05,9056.96,1.3,47.50,521.97,2388.03,8130.10,8.4441,0.03,393,2388,100.0,39.08,23.4166
3,1,4,0.0042,0.0000,100.0,518.67,642.44,1584.12,1406.42,14.62,21.61,554.07,2388.03,9045.29,1.3,47.28,521.38,2388.05,8132.90,8.3917,0.03,391,2388,100.0,39.00,23.3737
4,1,5,0.0014,0.0000,100.0,518.67,642.51,1587.19,1401.92,14.62,21.61,554.16,2388.01,9044.55,1.3,47.31,522.15,2388.03,8129.54,8.4031,0.03,390,2388,100.0,38.99,23.4130


In [6]:
df_test_rul_raw.head()

,RUL
0,112
1,98
2,69
3,82
4,91


# RUL Target Construction

The training trajectories run until failure. Therefore, the maximum observed cycle of each engine represents the failure point.

The raw RUL target is calculated as:

$$
RUL = \text{max cycle per engine} - \text{current cycle}
$$

A capped RUL target is created as a second target variable. This reflects the common piecewise linear degradation assumption that early engine life does not necessarily show observable degradation.

In [7]:
df_train_rul = add_rul_to_training_data(df_train_raw)

df_train_rul = add_capped_rul(
    df_train_rul,
    cap=RUL_CAP,
    source_col="RUL",
    target_col="RUL_capped"
)

df_train_rul[["engine", "cycle", "max_cycle", "RUL", "RUL_capped"]].head(100)


,engine,cycle,max_cycle,RUL,RUL_capped
0,1,1,192,191,125
1,1,2,192,190,125
2,1,3,192,189,125
3,1,4,192,188,125
4,1,5,192,187,125
...,...,...,...,...,...
95,1,96,192,96,96
96,1,97,192,95,95
97,1,98,192,94,94
98,1,99,192,93,93


In [8]:
df_train_rul[["RUL", "RUL_capped"]].describe()

,RUL,RUL_capped
count,20631.000000,20631.000000
mean,107.807862,86.829286
std,68.880990,41.673699
min,0.000000,0.000000
25%,51.000000,51.000000
50%,103.000000,103.000000
75%,155.000000,125.000000
max,361.000000,125.000000


# Test RUL Preparation

The test trajectories are incomplete and end before failure. The provided RUL file contains the true remaining useful life for the last observed cycle of each test engine.

This information is linked to the final observed test cycle in order to estimate the corresponding failure cycle for each test engine.

In [9]:
df_test_rul_summary = prepare_test_rul(
    df_test_raw,
    df_test_rul_raw
)

df_test_rul_summary.head()

,engine,last_observed_cycle,RUL,estimated_failure_cycle
0,1,31,112,143
1,2,49,98,147
2,3,126,69,195
3,4,106,82,188
4,5,98,91,189


In [10]:
print(f"Number of test engines: {df_test_raw['engine'].nunique()}")
print(f"Number of RUL values:   {df_test_rul_raw.shape[0]}")
print(f"Number of summary rows: {df_test_rul_summary.shape[0]}")

Number of test engines: 100
Number of RUL values:   100
Number of summary rows: 100


In [11]:
# Checks if every test engine has exactly one entry
assert df_test_rul_summary["engine"].nunique() == df_test_raw["engine"].nunique()

# Checks for missing RUL values
assert df_test_rul_summary["RUL"].notna().all()

# Checks for estimated failure cycle is after last obeserved cycle
assert (
    df_test_rul_summary["estimated_failure_cycle"]
    > df_test_rul_summary["last_observed_cycle"]
).all()

# Constant Feature Removal

Constant features do not provide useful information for RUL prediction within the selected subset. Based on the training data, constant operational settings and sensor measurements are identified and removed from both training and test data.

The identification is based only on the training data to avoid using information from the test set during preprocessing.

In [12]:
candidate_feature_cols = SETTING_NAMES + SENSOR_NAMES

constant_features = [
    feature
    for feature in candidate_feature_cols
    if df_train_raw[feature].nunique() == 1
]

constant_features

['setting_3',
 'T2 (Fan inlet temperature)',
 'P2 (Fan inlet pressure)',
 'epr (Engine pressure ratio (P50/P2))',
 'farB (Fuel-air ratio)',
 'Nf_dmd (Demanded fan speed)',
 'PCNfr_dmd (Demanded corrected fan speed)']

In [13]:
constant_feature_summary = pd.DataFrame({
    "feature": constant_features,
    "type": [
        "setting" if feature in SETTING_NAMES else "sensor"
        for feature in constant_features
    ],
    "constant_value": [
        df_train_raw[feature].iloc[0]
        for feature in constant_features
    ]
})

constant_feature_summary

,feature,type,constant_value
0,setting_3,setting,100.00
1,T2 (Fan inlet temperature),sensor,518.67
2,P2 (Fan inlet pressure),sensor,14.62
3,epr (Engine pressure ratio (P50/P2)),sensor,1.30
4,farB (Fuel-air ratio),sensor,0.03
5,Nf_dmd (Demanded fan speed),sensor,2388.00
6,PCNfr_dmd (Demanded corrected fan speed),sensor,100.00


In [14]:
df_train_preprocessed = df_train_rul.drop(columns=constant_features)
df_test_preprocessed = df_test_raw.drop(columns=constant_features)

In [15]:
removed_features_present_in_train = [
    feature for feature in constant_features
    if feature in df_train_preprocessed.columns
]

removed_features_present_in_test = [
    feature for feature in constant_features
    if feature in df_test_preprocessed.columns
]

assert len(removed_features_present_in_train) == 0
assert len(removed_features_present_in_test) == 0

In [16]:
print(f"Removed {len(constant_features)} constant features.")
print(f"Training shape before removal: {df_train_rul.shape}")
print(f"Training shape after removal:  {df_train_preprocessed.shape}")
print(f"Test shape before removal:     {df_test_raw.shape}")
print(f"Test shape after removal:      {df_test_preprocessed.shape}")

Removed 7 constant features.
Training shape before removal: (20631, 29)
Training shape after removal:  (20631, 22)
Test shape before removal:     (13096, 26)
Test shape after removal:      (13096, 19)


# Feature and Target Definition

After removing constant features, the remaining operational settings and sensor measurements are used as input features. Index columns and target columns are kept separate to avoid data leakage during model training.

In [17]:
feature_cols = [
    col for col in df_train_preprocessed.columns
    if col not in index_cols + target_cols + helper_cols
]

print(f"Number of feature columns: {len(feature_cols)}")
print(f"Number of target columns:  {len(target_cols)}")
print(f"Number of index columns:   {len(index_cols)}")

feature_cols

Number of feature columns: 17
Number of target columns:  2
Number of index columns:   2


['setting_1',
 'setting_2',
 'T24 (LPC outlet temperature)',
 'T30 (HPC outlet temperature)',
 'T50 (LPT outlet temperature)',
 'P15 (bypass duct pressure)',
 'P30 (HPC outlet pressure)',
 'Nf (Physical fan speed)',
 'Nc (Physical core speed)',
 'Ps30 (Static pressure at HPC outlet)',
 'phi (Ratio of fuel flow to Ps30)',
 'NRf (Corrected fan speed)',
 'NRc (Corrected core speed)',
 'BPR (Bypass ratio)',
 'htBleed (Bleed enthalpy)',
 'W31 (HPT Cool air flow)',
 'W32 (LPT Cool air flow)']

In [18]:
missing_in_test = [
    col for col in feature_cols
    if col not in df_test_preprocessed.columns
]

extra_in_test = [
    col for col in df_test_preprocessed.columns
    if col not in feature_cols + index_cols
]

print(f"Missing feature columns in test set: {missing_in_test}")
print(f"Extra columns in test set:           {extra_in_test}")

Missing feature columns in test set: []
Extra columns in test set:           []


In [19]:
assert len(missing_in_test) == 0
assert len(extra_in_test) == 0

In [20]:
# Keep index_cols for slising windows and engine-wise splits
X_train_full = df_train_preprocessed[index_cols + feature_cols].copy()
y_train_full = df_train_preprocessed[index_cols + [target_col]].copy()

X_test_full = df_test_preprocessed[index_cols + feature_cols].copy()


In [21]:
print(f"X_train_full shape: {X_train_full.shape}") # contains engine, cycle plus feture columns
print(f"y_train_full shape: {y_train_full.shape}") # contains engine, cycle, RUL_capped
print(f"X_test_full shape:  {X_test_full.shape}") # contais same input columns as X_train_full

X_train_full shape: (20631, 19)
y_train_full shape: (20631, 3)
X_test_full shape:  (13096, 19)


# Engine-wise Train-Validation Split

The training data is split into training and validation sets by engine unit. This prevents cycles from the same engine trajectory from appearing in both sets and avoids data leakage during model evaluation.

In [22]:
from sklearn.model_selection import train_test_split

In [23]:
unique_engines = df_train_preprocessed["engine"].unique()

train_engines, val_engines = train_test_split(
    unique_engines,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True
)

print(f"Number of training engines:   {len(train_engines)}")
print(f"Number of validation engines: {len(val_engines)}")


Number of training engines:   80
Number of validation engines: 20


In [24]:
train_mask = df_train_preprocessed["engine"].isin(train_engines)
val_mask = df_train_preprocessed["engine"].isin(val_engines)

df_train_split = df_train_preprocessed[train_mask].copy()
df_val_split = df_train_preprocessed[val_mask].copy()

print(f"Train split shape:      {df_train_split.shape}")
print(f"Validation split shape: {df_val_split.shape}")

Train split shape:      (16561, 22)
Validation split shape: (4070, 22)


In [25]:
# Ensure that no engine appears in both the training and validation engine sets.
assert set(train_engines).isdisjoint(set(val_engines))

# Ensure that all selected training engines are present in the training split.
assert df_train_split["engine"].nunique() == len(train_engines)

# Ensure that all selected validation engines are present in the validation split.
assert df_val_split["engine"].nunique() == len(val_engines)

# Ensure that the split preserves all original training rows without dropping or duplicating observations.
assert df_train_split.shape[0] + df_val_split.shape[0] == df_train_preprocessed.shape[0]

In [26]:
X_train = df_train_split[index_cols + feature_cols].copy()
y_train = df_train_split[index_cols + [target_col]].copy()

X_val = df_val_split[index_cols + feature_cols].copy()
y_val = df_val_split[index_cols + [target_col]].copy()

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"y_val shape:   {y_val.shape}")

X_train shape: (16561, 19)
y_train shape: (16561, 3)
X_val shape:   (4070, 19)
y_val shape:   (4070, 3)


In [27]:
# Ensure that no engine appears in both the final training and validation feature sets.
assert set(X_train["engine"]).isdisjoint(set(X_val["engine"]))

# Ensure that every training feature row has a corresponding training target row.
assert X_train.shape[0] == y_train.shape[0]

# Ensure that every validation feature row has a corresponding validation target row.
assert X_val.shape[0] == y_val.shape[0]

The split was performed by engine unit rather than by individual rows. This ensures that complete degradation trajectories are assigned either to training or validation and prevents leakage between both sets.

# Feature Scaling

Numerical feature scaling is prepared for models that are sensitive to feature magnitude. The scaler is fitted only on the training split and then applied to the validation and test sets to avoid data leakage.

In [28]:
from sklearn.preprocessing import StandardScaler

In [29]:
scaler = StandardScaler()

scaler.fit(X_train[feature_cols])

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [30]:
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test_full.copy()

X_train_scaled[feature_cols] = scaler.transform(X_train[feature_cols])
X_val_scaled[feature_cols] = scaler.transform(X_val[feature_cols])
X_test_scaled[feature_cols] = scaler.transform(X_test_full[feature_cols])

In [31]:
X_train_scaled[feature_cols].describe().T[["mean", "std"]].head()

,mean,std
setting_1,0.000000e+00,1.00003
setting_2,-2.231038e-17,1.00003
T24 (LPC outlet temperature),-2.197573e-14,1.00003
T30 (HPC outlet temperature),-1.421858e-14,1.00003
T50 (LPT outlet temperature),-4.400294e-15,1.00003


In [32]:
# Ensure that scaling does not change the shape of the training features.
assert X_train_scaled.shape == X_train.shape

# Ensure that scaling does not change the shape of the validation features.
assert X_val_scaled.shape == X_val.shape

# Ensure that scaling does not change the shape of the test features.
assert X_test_scaled.shape == X_test_full.shape

# Ensure that engine and cycle identifiers remain unchanged in the scaled training set.
assert X_train_scaled[index_cols].equals(X_train[index_cols])

# Ensure that engine and cycle identifiers remain unchanged in the scaled validation set.
assert X_val_scaled[index_cols].equals(X_val[index_cols])

# Ensure that engine and cycle identifiers remain unchanged in the scaled test set.
assert X_test_scaled[index_cols].equals(X_test_full[index_cols])

In [33]:
scaler_summary = pd.DataFrame({
    "feature": feature_cols,
    "mean": scaler.mean_,
    "scale": scaler.scale_
})

scaler_summary.head()

,feature,mean,scale
0,setting_1,-0.000007,0.002191
1,setting_2,0.000002,0.000293
2,T24 (LPC outlet temperature),642.677606,0.498109
3,T30 (HPC outlet temperature),1590.489097,6.085225
4,T50 (LPT outlet temperature),1408.879098,8.965345


# Preprocessing Validation

The final preprocessing outputs are validated to ensure that feature columns are consistent, targets are available for training and validation, and no data leakage is introduced through the train-validation split.

In [34]:
# Overview over final objects
preprocessing_outputs = pd.DataFrame({
    "object": [
        "X_train",
        "y_train",
        "X_val",
        "y_val",
        "X_test_full",
        "X_train_scaled",
        "X_val_scaled",
        "X_test_scaled",
    ],
    "rows": [
        X_train.shape[0],
        y_train.shape[0],
        X_val.shape[0],
        y_val.shape[0],
        X_test_full.shape[0],
        X_train_scaled.shape[0],
        X_val_scaled.shape[0],
        X_test_scaled.shape[0],
    ],
    "columns": [
        X_train.shape[1],
        y_train.shape[1],
        X_val.shape[1],
        y_val.shape[1],
        X_test_full.shape[1],
        X_train_scaled.shape[1],
        X_val_scaled.shape[1],
        X_test_scaled.shape[1],
    ],
})

preprocessing_outputs

,object,rows,columns
0,X_train,16561,19
1,y_train,16561,3
2,X_val,4070,19
3,y_val,4070,3
4,X_test_full,13096,19
5,X_train_scaled,16561,19
6,X_val_scaled,4070,19
7,X_test_scaled,13096,19


In [35]:
# Check for missing values
missing_value_summary = pd.DataFrame({
    "object": [
        "X_train",
        "y_train",
        "X_val",
        "y_val",
        "X_test_full",
        "X_train_scaled",
        "X_val_scaled",
        "X_test_scaled",
    ],
    "missing_values": [
        X_train.isna().sum().sum(),
        y_train.isna().sum().sum(),
        X_val.isna().sum().sum(),
        y_val.isna().sum().sum(),
        X_test_full.isna().sum().sum(),
        X_train_scaled.isna().sum().sum(),
        X_val_scaled.isna().sum().sum(),
        X_test_scaled.isna().sum().sum(),
    ],
})

missing_value_summary

,object,missing_values
0,X_train,0
1,y_train,0
2,X_val,0
3,y_val,0
4,X_test_full,0
5,X_train_scaled,0
6,X_val_scaled,0
7,X_test_scaled,0


In [36]:
# Ensure that no missing values are present in the final feature and target datasets.
assert missing_value_summary["missing_values"].sum() == 0

In [37]:
# Ensure that the selected target column is available for training and validation.
assert target_col in y_train.columns
assert target_col in y_val.columns

# Ensure that target values are non-negative.
assert (y_train[target_col] >= 0).all()
assert (y_val[target_col] >= 0).all()

# Ensure that capped RUL values do not exceed the defined cap.
assert y_train[target_col].max() <= RUL_CAP
assert y_val[target_col].max() <= RUL_CAP

In [38]:
# Ensure that removed constant features are not part of the final feature set.
assert not any(feature in feature_cols for feature in constant_features)

# Ensure that all final feature columns are present in train, validation and test.
assert all(feature in X_train.columns for feature in feature_cols)
assert all(feature in X_val.columns for feature in feature_cols)
assert all(feature in X_test_full.columns for feature in feature_cols)

In [39]:
print("Preprocessing validation completed successfully.")
print(f"Final number of input features: {len(feature_cols)}")
print(f"Target column: {target_col}")
print(f"Removed constant features: {len(constant_features)}")
print(f"Training engines: {X_train['engine'].nunique()}")
print(f"Validation engines: {X_val['engine'].nunique()}")
print(f"Test engines: {X_test_full['engine'].nunique()}")

Preprocessing validation completed successfully.
Final number of input features: 17
Target column: RUL_capped
Removed constant features: 7
Training engines: 80
Validation engines: 20
Test engines: 100


# Preprocessing Takeaways

This notebook transformed the selected C-MAPSS subset into a structured preprocessing output for later feature engineering and model training.

The raw training trajectories were enriched with two target variables: the original `RUL` target and the capped `RUL_capped` target. The capped target is used as the primary modeling target because it reflects the common assumption that early-life degradation is not always directly observable.

The test RUL file was linked to the final observed cycle of each test engine. This provides the true remaining useful life at the end of each test trajectory and prepares the test set for later model evaluation.

Constant features were identified using only the training data and removed from both training and test features. This avoids retaining signals that do not provide useful information for RUL prediction within the selected subset.

The feature columns, target columns and index columns were explicitly separated. This prevents target leakage and ensures that helper variables such as `max_cycle`, `RUL` and `RUL_capped` are not used as model input features.

The training data was split by engine unit into training and validation sets. This prevents cycles from the same engine trajectory from appearing in both sets and avoids data leakage during validation.

Scaled feature versions were created by fitting the scaler only on the training split and applying it to validation and test data. Both unscaled and scaled feature sets are kept because different model families require different preprocessing assumptions.

The final preprocessed outputs are ready for Notebook 04, where sliding-window datasets and handcrafted time-series features will be created.

The main outputs of this notebook are:

- `X_train`, `y_train`
- `X_val`, `y_val`
- `X_test_full`
- `X_train_scaled`, `X_val_scaled`, `X_test_scaled`
- `feature_cols`
- `target_col`
- `constant_features`
- `df_test_rul_summary`